# Day 1 下午：進階語音助理 Agent 實作

**LLM Application - AI Agent 課程**

本單元將帶你實作五種進階 Agent 設計模式，並整合成一個完整的語音助理系統：

1. 語音助理架構規劃
2. Pattern 1: Pipeline 管線式流程
3. Pattern 2: Orchestrator 編排調度模式
4. Pattern 3: Memory-Augmented 記憶增強模式
5. Pattern 4: Human-in-the-loop 人機協作審核
6. Pattern 5: Tool-Augmented 工具增強模式
7. 整合五種模式：打造進階語音助理 Agent
8. 實作練習

In [ ]:
%%capture
!pip install openai langchain langchain-openai langgraph

In [ ]:
import os

# 請填入你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from typing import TypedDict, Optional
from langchain_openai import ChatOpenAI
from openai import OpenAI
from langgraph.graph import StateGraph, END
from IPython.display import Image, display, Audio
import json

# 初始化 LLM 與 OpenAI client
llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)
client = OpenAI()

---
## 1. 語音助理架構規劃

進階語音助理不再只是「聽 → 回」的簡單流程，而是結合多種 Agent 設計模式的複合系統。

### 架構總覽

```
使用者語音輸入
    │
    ▼
┌─────────────────────────────────────────────────┐
│  Pattern 1: Pipeline 管線式流程                    │
│  STT → 正規化 → 意圖分類 → 回應生成 → TTS          │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  Pattern 2: Orchestrator 編排調度                  │
│  根據意圖分類結果，動態決定後續流程                    │
│  ├─ 查詢資訊 → info_node                          │
│  ├─ 客訴處理 → complaint_node                     │
│  └─ 一般對話 → general_node                       │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  Pattern 3: Memory-Augmented 記憶增強              │
│  載入對話歷史 → 處理 → 儲存新的對話記錄               │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  Pattern 4: Human-in-the-loop 人機協作             │
│  信心度檢查 → 低於門檻 → 轉人工審核                   │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  Pattern 5: Tool-Augmented 工具增強                │
│  工具選擇 → 工具執行 → 結果整合                      │
└─────────────────────────────────────────────────┘
    │
    ▼
語音回應輸出
```

### 五種模式如何組合

- **Pipeline** 是骨架，定義基本的處理流程
- **Orchestrator** 根據意圖做動態路由
- **Memory** 讓助理能記住上下文，實現多輪對話
- **Human-in-the-loop** 在信心不足時引入人工審核
- **Tool-Augmented** 讓助理能呼叫外部工具完成任務

接下來我們會逐一實作每個模式，最後整合成完整系統。

---
## 2. Pattern 1: Pipeline 管線式流程

最基本的模式：將語音助理拆成一條線性管線，每個節點負責一件事。

```
STT → 正規化 → 意圖分類 → 回應生成 → TTS
```

In [ ]:
# 定義進階語音助理的 State
class AdvancedVoiceState(TypedDict):
    audio_input: Optional[str]       # 語音輸入（檔案路徑或 base64）
    transcript: Optional[str]        # STT 轉錄文字
    normalized_text: Optional[str]   # 正規化後的文字
    intent: Optional[str]            # 意圖分類結果
    context: Optional[str]           # 上下文資訊
    response_text: Optional[str]     # 回應文字
    audio_output: Optional[str]      # TTS 語音輸出
    tools_result: Optional[str]      # 工具呼叫結果
    memory: Optional[list]           # 對話記憶
    needs_human: Optional[bool]      # 是否需要人工審核
    confidence: Optional[float]      # 信心度分數

In [ ]:
# --- 節點函式 ---

# STT 節點：語音轉文字
def stt_node(state: AdvancedVoiceState) -> dict:
    """語音轉文字（這裡用模擬的方式）"""
    audio = state.get("audio_input", "")
    # 實際場景會用 OpenAI Whisper API：
    # transcript = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
    # 這裡直接用文字模擬
    print(f"[STT] 收到音訊，轉錄中...")
    transcript = audio  # 模擬：直接把輸入當作轉錄結果
    print(f"[STT] 轉錄結果: {transcript}")
    return {"transcript": transcript}


# 正規化節點：清理文字
def normalize_node(state: AdvancedVoiceState) -> dict:
    """正規化文字：去除多餘空白、統一格式"""
    text = state.get("transcript", "")
    normalized = text.strip().replace("  ", " ")
    print(f"[正規化] {text} → {normalized}")
    return {"normalized_text": normalized}


# 意圖分類節點
def classify_intent_node(state: AdvancedVoiceState) -> dict:
    """用 LLM 分類使用者意圖"""
    text = state.get("normalized_text", "")
    prompt = f"""請分析以下使用者輸入的意圖，只回傳以下其中一個分類：
- query_info（查詢資訊，例如查詢訂單、餘額）
- complaint（客訴、抱怨）
- general（一般對話、閒聊）

使用者輸入：{text}

只回傳分類名稱，不要其他文字。"""
    result = llm.invoke(prompt)
    intent = result.content.strip().lower()
    print(f"[意圖分類] '{text}' → {intent}")
    return {"intent": intent}


# 回應生成節點
def response_node(state: AdvancedVoiceState) -> dict:
    """根據意圖生成回應"""
    text = state.get("normalized_text", "")
    intent = state.get("intent", "general")
    prompt = f"""你是一個友善的客服語音助理。
使用者意圖：{intent}
使用者說：{text}

請用繁體中文回應，語氣自然親切，回應簡潔。"""
    result = llm.invoke(prompt)
    response = result.content.strip()
    print(f"[回應生成] {response}")
    return {"response_text": response}


# TTS 節點：文字轉語音
def tts_node(state: AdvancedVoiceState) -> dict:
    """文字轉語音（模擬）"""
    text = state.get("response_text", "")
    # 實際場景會用 OpenAI TTS API：
    # speech = client.audio.speech.create(model="tts-1", voice="alloy", input=text)
    print(f"[TTS] 正在將回應轉為語音...")
    print(f"[TTS] 語音內容: {text}")
    return {"audio_output": f"audio_of({text[:30]}...)"}

print("所有 Pipeline 節點定義完成")

In [ ]:
# 建立 Pipeline 圖
pipeline_graph = StateGraph(AdvancedVoiceState)

# 加入節點
pipeline_graph.add_node("stt", stt_node)
pipeline_graph.add_node("normalize", normalize_node)
pipeline_graph.add_node("classify_intent", classify_intent_node)
pipeline_graph.add_node("response", response_node)
pipeline_graph.add_node("tts", tts_node)

# 線性連接
pipeline_graph.set_entry_point("stt")
pipeline_graph.add_edge("stt", "normalize")
pipeline_graph.add_edge("normalize", "classify_intent")
pipeline_graph.add_edge("classify_intent", "response")
pipeline_graph.add_edge("response", "tts")
pipeline_graph.add_edge("tts", END)

# 編譯
pipeline_app = pipeline_graph.compile()
print("Pipeline 圖建立完成")

In [ ]:
# 視覺化 Pipeline
try:
    display(Image(pipeline_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"視覺化需要額外套件，跳過: {e}")

In [ ]:
# 測試 Pipeline
result = pipeline_app.invoke({
    "audio_input": "我想查一下我的訂單狀態",
})
print("\n=== Pipeline 結果 ===")
print(f"意圖: {result['intent']}")
print(f"回應: {result['response_text']}")

---
## 3. Pattern 2: Orchestrator 編排調度模式

加入一個 **Orchestrator（編排器）** 節點，根據意圖分類結果，動態決定下一步要走哪個處理節點。

```
意圖分類 → Orchestrator → ┬─ query_info → info_node
                           ├─ complaint  → complaint_node
                           └─ general    → general_node
```

In [ ]:
# --- Orchestrator 模式的專用節點 ---

# 資訊查詢節點
def info_node(state: AdvancedVoiceState) -> dict:
    """處理資訊查詢類的請求"""
    text = state.get("normalized_text", "")
    prompt = f"""你是專業的客服助理，負責處理資訊查詢。
使用者問：{text}

請用繁體中文回應，提供具體的查詢指引。"""
    result = llm.invoke(prompt)
    print(f"[Info Node] 處理資訊查詢")
    return {"response_text": result.content.strip()}


# 客訴處理節點
def complaint_node(state: AdvancedVoiceState) -> dict:
    """處理客訴類的請求"""
    text = state.get("normalized_text", "")
    prompt = f"""你是專業的客訴處理人員，要先同理使用者的感受，再提出解決方案。
使用者說：{text}

請用繁體中文回應，語氣溫和有同理心。"""
    result = llm.invoke(prompt)
    print(f"[Complaint Node] 處理客訴")
    return {"response_text": result.content.strip()}


# 一般對話節點
def general_node(state: AdvancedVoiceState) -> dict:
    """處理一般閒聊"""
    text = state.get("normalized_text", "")
    prompt = f"""你是友善的語音助理，正在跟使用者閒聊。
使用者說：{text}

請用繁體中文回應，簡短自然。"""
    result = llm.invoke(prompt)
    print(f"[General Node] 處理一般對話")
    return {"response_text": result.content.strip()}


# Orchestrator 路由函式
def orchestrator_route(state: AdvancedVoiceState) -> str:
    """根據意圖決定下一個節點"""
    intent = state.get("intent", "general")
    print(f"[Orchestrator] 意圖={intent}，路由中...")
    if "query" in intent or "info" in intent:
        return "info_node"
    elif "complaint" in intent:
        return "complaint_node"
    else:
        return "general_node"

print("Orchestrator 節點定義完成")

In [ ]:
# 建立 Orchestrator 圖
orchestrator_graph = StateGraph(AdvancedVoiceState)

# 加入節點
orchestrator_graph.add_node("stt", stt_node)
orchestrator_graph.add_node("normalize", normalize_node)
orchestrator_graph.add_node("classify_intent", classify_intent_node)
orchestrator_graph.add_node("info_node", info_node)
orchestrator_graph.add_node("complaint_node", complaint_node)
orchestrator_graph.add_node("general_node", general_node)
orchestrator_graph.add_node("tts", tts_node)

# 設定流程
orchestrator_graph.set_entry_point("stt")
orchestrator_graph.add_edge("stt", "normalize")
orchestrator_graph.add_edge("normalize", "classify_intent")

# 條件邊：根據意圖路由
orchestrator_graph.add_conditional_edges(
    "classify_intent",
    orchestrator_route,
    {
        "info_node": "info_node",
        "complaint_node": "complaint_node",
        "general_node": "general_node",
    }
)

# 所有處理節點都連到 TTS
orchestrator_graph.add_edge("info_node", "tts")
orchestrator_graph.add_edge("complaint_node", "tts")
orchestrator_graph.add_edge("general_node", "tts")
orchestrator_graph.add_edge("tts", END)

orchestrator_app = orchestrator_graph.compile()
print("Orchestrator 圖建立完成")

In [ ]:
# 視覺化 Orchestrator
try:
    display(Image(orchestrator_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"視覺化需要額外套件，跳過: {e}")

In [ ]:
# 測試不同意圖的路由
test_inputs = [
    "我想查一下我的訂單到哪了",
    "你們的服務爛透了，我要投訴",
    "今天天氣真好啊",
]

for user_input in test_inputs:
    print(f"\n{'='*60}")
    print(f"使用者: {user_input}")
    print(f"{'='*60}")
    result = orchestrator_app.invoke({"audio_input": user_input})
    print(f"\n回應: {result['response_text'][:100]}...")

---
## 4. Pattern 3: Memory-Augmented 記憶增強模式

讓語音助理能記住先前的對話，實現多輪對話能力。

使用 state 中的 `memory` 欄位（list）來儲存對話歷史。

```
記憶載入 → 處理 → 記憶儲存
```

In [ ]:
# --- Memory 模式的節點 ---

# 記憶載入節點
def memory_load_node(state: AdvancedVoiceState) -> dict:
    """載入對話歷史"""
    memory = state.get("memory", []) or []
    print(f"[Memory Load] 載入 {len(memory)} 筆對話記錄")
    if memory:
        # 把記憶整理成上下文字串
        context = "\n".join([f"{m['role']}: {m['content']}" for m in memory[-6:]])  # 最近 6 筆
        return {"context": context}
    return {"context": "（無歷史對話）"}


# 記憶儲存節點
def memory_save_node(state: AdvancedVoiceState) -> dict:
    """將本輪對話存入記憶"""
    memory = state.get("memory", []) or []
    user_text = state.get("normalized_text", "")
    response = state.get("response_text", "")
    # 加入新的對話記錄
    new_memory = memory + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": response},
    ]
    print(f"[Memory Save] 儲存完成，共 {len(new_memory)} 筆記錄")
    return {"memory": new_memory}


# 有記憶的回應生成節點
def response_with_memory_node(state: AdvancedVoiceState) -> dict:
    """根據記憶和當前輸入生成回應"""
    text = state.get("normalized_text", "")
    context = state.get("context", "")
    prompt = f"""你是一個友善的客服語音助理，能記住先前的對話。

對話歷史：
{context}

使用者現在說：{text}

請根據對話歷史和當前輸入，用繁體中文回應。保持對話連貫性。"""
    result = llm.invoke(prompt)
    response = result.content.strip()
    print(f"[回應生成(含記憶)] {response[:80]}...")
    return {"response_text": response}

print("Memory 節點定義完成")

In [ ]:
# 建立 Memory-Augmented 圖
memory_graph = StateGraph(AdvancedVoiceState)

memory_graph.add_node("stt", stt_node)
memory_graph.add_node("normalize", normalize_node)
memory_graph.add_node("memory_load", memory_load_node)
memory_graph.add_node("response", response_with_memory_node)
memory_graph.add_node("memory_save", memory_save_node)
memory_graph.add_node("tts", tts_node)

memory_graph.set_entry_point("stt")
memory_graph.add_edge("stt", "normalize")
memory_graph.add_edge("normalize", "memory_load")
memory_graph.add_edge("memory_load", "response")
memory_graph.add_edge("response", "memory_save")
memory_graph.add_edge("memory_save", "tts")
memory_graph.add_edge("tts", END)

memory_app = memory_graph.compile()
print("Memory-Augmented 圖建立完成")

In [ ]:
# 測試多輪對話（模擬記憶傳遞）
conversation = [
    "我叫小明，我想查訂單",
    "訂單編號是 A12345",
    "那我的另一筆訂單 B67890 呢？",
]

memory = []  # 初始記憶為空

for user_input in conversation:
    print(f"\n{'='*60}")
    print(f"使用者: {user_input}")
    print(f"{'='*60}")
    result = memory_app.invoke({
        "audio_input": user_input,
        "memory": memory,  # 傳入上一輪的記憶
    })
    memory = result["memory"]  # 更新記憶
    print(f"\n回應: {result['response_text'][:150]}")
    print(f"記憶筆數: {len(memory)}")

---
## 5. Pattern 4: Human-in-the-loop 人機協作審核

當 LLM 對自己的回應信心度不夠時，將對話轉交給人工審核。

```
回應生成 → 信心度檢查 → ┬─ 高信心 → 直接回覆
                         └─ 低信心 → 人工審核 → 回覆
```

In [ ]:
# --- Human-in-the-loop 模式的節點 ---

# 信心度檢查節點
def confidence_check_node(state: AdvancedVoiceState) -> dict:
    """用 LLM 評估回應的信心度"""
    text = state.get("normalized_text", "")
    response = state.get("response_text", "")
    prompt = f"""請評估以下客服回應的信心度（0.0 到 1.0）。

使用者問題：{text}
客服回應：{response}

評估標準：
- 回應是否完整且正確？
- 是否有模糊或不確定的表達？
- 是否需要人工確認？

只回傳一個數字（0.0 到 1.0），不要其他文字。"""
    result = llm.invoke(prompt)
    try:
        confidence = float(result.content.strip())
    except:
        confidence = 0.5  # 解析失敗時給中等信心度
    
    needs_human = confidence < 0.7  # 低於 0.7 就需要人工審核
    print(f"[信心度檢查] 分數={confidence:.2f}, 需人工={needs_human}")
    return {"confidence": confidence, "needs_human": needs_human}


# 人工審核節點
def human_review_node(state: AdvancedVoiceState) -> dict:
    """人工審核（模擬）"""
    response = state.get("response_text", "")
    print(f"[人工審核] 原始回應: {response[:100]}")
    print(f"[人工審核] ⚠️ 此回應信心度不足，已轉交人工處理")
    # 實際場景可用 input() 讓人工輸入修正：
    # corrected = input("請人工輸入修正後的回應: ")
    # 這裡用模擬的方式
    corrected = f"[人工審核後] {response}"
    return {"response_text": corrected}


# 路由函式：根據信心度決定
def confidence_route(state: AdvancedVoiceState) -> str:
    """根據信心度決定是否需要人工審核"""
    needs_human = state.get("needs_human", False)
    if needs_human:
        return "human_review"
    return "tts"

print("Human-in-the-loop 節點定義完成")

In [ ]:
# 建立 Human-in-the-loop 圖
hitl_graph = StateGraph(AdvancedVoiceState)

hitl_graph.add_node("stt", stt_node)
hitl_graph.add_node("normalize", normalize_node)
hitl_graph.add_node("response", response_node)
hitl_graph.add_node("confidence_check", confidence_check_node)
hitl_graph.add_node("human_review", human_review_node)
hitl_graph.add_node("tts", tts_node)

hitl_graph.set_entry_point("stt")
hitl_graph.add_edge("stt", "normalize")
hitl_graph.add_edge("normalize", "response")
hitl_graph.add_edge("response", "confidence_check")

# 條件邊：信心度高就直接 TTS，低就轉人工
hitl_graph.add_conditional_edges(
    "confidence_check",
    confidence_route,
    {
        "human_review": "human_review",
        "tts": "tts",
    }
)

hitl_graph.add_edge("human_review", "tts")
hitl_graph.add_edge("tts", END)

hitl_app = hitl_graph.compile()
print("Human-in-the-loop 圖建立完成")

In [ ]:
# 視覺化 Human-in-the-loop
try:
    display(Image(hitl_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"視覺化需要額外套件，跳過: {e}")

In [ ]:
# 測試 Human-in-the-loop
test_inputs = [
    "你好",  # 簡單問題，信心度應該高
    "我三個月前買的東西壞了，但保固期限不確定，你們的退換貨政策是什麼？",  # 複雜問題，信心度可能低
]

for user_input in test_inputs:
    print(f"\n{'='*60}")
    print(f"使用者: {user_input}")
    print(f"{'='*60}")
    result = hitl_app.invoke({"audio_input": user_input})
    print(f"\n最終回應: {result['response_text'][:150]}")
    print(f"信心度: {result.get('confidence', 'N/A')}")
    print(f"經人工審核: {result.get('needs_human', False)}")

---
## 6. Pattern 5: Tool-Augmented 工具增強模式

讓語音助理能呼叫外部工具（查詢訂單、查餘額、轉接真人客服等）。

```
意圖分類 → 工具選擇 → 工具執行 → 結果整合 → 回應
```

In [ ]:
# --- 定義工具 ---

def query_order_status(order_id: str) -> str:
    """查詢訂單狀態（模擬）"""
    fake_orders = {
        "A12345": {"status": "已出貨", "eta": "2024-12-25"},
        "B67890": {"status": "處理中", "eta": "2024-12-28"},
    }
    if order_id in fake_orders:
        info = fake_orders[order_id]
        return f"訂單 {order_id}: 狀態={info['status']}, 預計到貨={info['eta']}"
    return f"查無訂單 {order_id}"


def check_account_balance(account_id: str) -> str:
    """查詢帳戶餘額（模擬）"""
    fake_accounts = {
        "user001": 15000,
        "user002": 3200,
    }
    balance = fake_accounts.get(account_id, None)
    if balance is not None:
        return f"帳戶 {account_id} 餘額: NT${balance:,}"
    return f"查無帳戶 {account_id}"


def transfer_to_agent(reason: str) -> str:
    """轉接真人客服（模擬）"""
    return f"已將您轉接至真人客服，原因：{reason}。請稍候..."


# 工具清單
TOOLS = {
    "query_order_status": query_order_status,
    "check_account_balance": check_account_balance,
    "transfer_to_agent": transfer_to_agent,
}

print(f"已定義 {len(TOOLS)} 個工具: {list(TOOLS.keys())}")

In [ ]:
# --- Tool-Augmented 節點 ---

# 工具選擇節點
def tool_selection_node(state: AdvancedVoiceState) -> dict:
    """用 LLM 決定要使用哪個工具"""
    text = state.get("normalized_text", "")
    intent = state.get("intent", "")
    prompt = f"""根據使用者的意圖和輸入，決定要使用哪個工具。

可用工具：
1. query_order_status - 查詢訂單狀態，需要參數 order_id
2. check_account_balance - 查詢帳戶餘額，需要參數 account_id
3. transfer_to_agent - 轉接真人客服，需要參數 reason
4. none - 不需要工具

使用者意圖：{intent}
使用者輸入：{text}

請回傳 JSON 格式：
{{"tool": "工具名稱", "params": {{"參數名": "參數值"}}}}

如果不需要工具，回傳：
{{"tool": "none", "params": {{}}}}

只回傳 JSON，不要其他文字。"""
    result = llm.invoke(prompt)
    try:
        # 清理可能的 markdown 格式
        content = result.content.strip()
        content = content.replace("```json", "").replace("```", "").strip()
        tool_decision = json.loads(content)
    except:
        tool_decision = {"tool": "none", "params": {}}
    
    print(f"[工具選擇] 決定使用: {tool_decision}")
    return {"tools_result": json.dumps(tool_decision, ensure_ascii=False)}


# 工具執行節點
def tool_execution_node(state: AdvancedVoiceState) -> dict:
    """執行選定的工具"""
    tools_result = state.get("tools_result", "{}")
    try:
        tool_decision = json.loads(tools_result)
    except:
        tool_decision = {"tool": "none", "params": {}}
    
    tool_name = tool_decision.get("tool", "none")
    params = tool_decision.get("params", {})
    
    if tool_name in TOOLS:
        # 呼叫對應的工具
        result = TOOLS[tool_name](**params)
        print(f"[工具執行] {tool_name}({params}) → {result}")
        return {"tools_result": result}
    else:
        print(f"[工具執行] 不需要工具")
        return {"tools_result": "不需要工具"}


# 整合工具結果的回應節點
def response_with_tools_node(state: AdvancedVoiceState) -> dict:
    """整合工具結果生成回應"""
    text = state.get("normalized_text", "")
    tools_result = state.get("tools_result", "")
    prompt = f"""你是友善的客服語音助理。

使用者說：{text}
工具查詢結果：{tools_result}

請根據工具查詢結果，用繁體中文回覆使用者。語氣自然親切。"""
    result = llm.invoke(prompt)
    response = result.content.strip()
    print(f"[回應生成(含工具)] {response[:80]}")
    return {"response_text": response}

print("Tool-Augmented 節點定義完成")

In [ ]:
# 建立 Tool-Augmented 圖
tool_graph = StateGraph(AdvancedVoiceState)

tool_graph.add_node("stt", stt_node)
tool_graph.add_node("normalize", normalize_node)
tool_graph.add_node("classify_intent", classify_intent_node)
tool_graph.add_node("tool_selection", tool_selection_node)
tool_graph.add_node("tool_execution", tool_execution_node)
tool_graph.add_node("response", response_with_tools_node)
tool_graph.add_node("tts", tts_node)

tool_graph.set_entry_point("stt")
tool_graph.add_edge("stt", "normalize")
tool_graph.add_edge("normalize", "classify_intent")
tool_graph.add_edge("classify_intent", "tool_selection")
tool_graph.add_edge("tool_selection", "tool_execution")
tool_graph.add_edge("tool_execution", "response")
tool_graph.add_edge("response", "tts")
tool_graph.add_edge("tts", END)

tool_app = tool_graph.compile()
print("Tool-Augmented 圖建立完成")

In [ ]:
# 測試 Tool-Augmented
test_inputs = [
    "幫我查訂單 A12345 的狀態",
    "我想看 user001 的帳戶餘額",
    "我要找真人客服",
]

for user_input in test_inputs:
    print(f"\n{'='*60}")
    print(f"使用者: {user_input}")
    print(f"{'='*60}")
    result = tool_app.invoke({"audio_input": user_input})
    print(f"\n回應: {result['response_text'][:150]}")

---
## 7. 整合五種模式：打造進階語音助理 Agent

現在把五種模式全部整合成一個完整的語音助理 Agent。

### 完整流程

```
STT → 正規化 → 記憶載入 → 意圖分類 → Orchestrator
                                        │
                           ┌─────────────┼─────────────┐
                           ▼             ▼             ▼
                      info_path     complaint_path  general_path
                      (工具選擇→     (直接回應)      (直接回應)
                       工具執行→
                       回應)
                           │             │             │
                           └─────────────┼─────────────┘
                                         ▼
                                    信心度檢查
                                    ┌────┴────┐
                                    ▼         ▼
                               直接回覆   人工審核
                                    │         │
                                    └────┬────┘
                                         ▼
                                    記憶儲存 → TTS
```

In [ ]:
# === 完整整合版 State ===
class FullVoiceState(TypedDict):
    audio_input: Optional[str]
    transcript: Optional[str]
    normalized_text: Optional[str]
    intent: Optional[str]
    context: Optional[str]
    response_text: Optional[str]
    audio_output: Optional[str]
    tools_result: Optional[str]
    memory: Optional[list]
    needs_human: Optional[bool]
    confidence: Optional[float]

print("完整 State 定義完成")

In [ ]:
# === 整合版專用節點 ===

# 完整版 Orchestrator 路由
def full_orchestrator_route(state: FullVoiceState) -> str:
    intent = state.get("intent", "general")
    print(f"[Full Orchestrator] 意圖={intent}")
    if "query" in intent or "info" in intent:
        return "tool_selection"  # 查詢類走工具路線
    elif "complaint" in intent:
        return "complaint_response"
    else:
        return "general_response"


# 整合版：客訴回應
def full_complaint_response(state: FullVoiceState) -> dict:
    text = state.get("normalized_text", "")
    context = state.get("context", "")
    prompt = f"""你是專業的客訴處理人員。
對話歷史：{context}
使用者說：{text}

先同理使用者感受，再提出解決方案。用繁體中文回應。"""
    result = llm.invoke(prompt)
    print(f"[客訴回應] 生成完成")
    return {"response_text": result.content.strip()}


# 整合版：一般回應
def full_general_response(state: FullVoiceState) -> dict:
    text = state.get("normalized_text", "")
    context = state.get("context", "")
    prompt = f"""你是友善的語音助理。
對話歷史：{context}
使用者說：{text}

用繁體中文回應，簡短自然。"""
    result = llm.invoke(prompt)
    print(f"[一般回應] 生成完成")
    return {"response_text": result.content.strip()}


# 整合版：工具選擇（複用前面的邏輯）
def full_tool_selection(state: FullVoiceState) -> dict:
    text = state.get("normalized_text", "")
    prompt = f"""根據使用者輸入，決定要使用哪個工具。

可用工具：
1. query_order_status - 查詢訂單狀態，需要 order_id
2. check_account_balance - 查詢帳戶餘額，需要 account_id
3. transfer_to_agent - 轉接真人客服，需要 reason
4. none - 不需要工具

使用者輸入：{text}

回傳 JSON：{{"tool": "工具名稱", "params": {{"參數名": "參數值"}}}}
只回傳 JSON。"""
    result = llm.invoke(prompt)
    content = result.content.strip().replace("```json", "").replace("```", "").strip()
    try:
        tool_decision = json.loads(content)
    except:
        tool_decision = {"tool": "none", "params": {}}
    print(f"[工具選擇] {tool_decision}")
    return {"tools_result": json.dumps(tool_decision, ensure_ascii=False)}


# 整合版：工具執行
def full_tool_execution(state: FullVoiceState) -> dict:
    tools_result = state.get("tools_result", "{}")
    try:
        tool_decision = json.loads(tools_result)
    except:
        return {"tools_result": "工具解析失敗"}
    
    tool_name = tool_decision.get("tool", "none")
    params = tool_decision.get("params", {})
    if tool_name in TOOLS:
        result = TOOLS[tool_name](**params)
        print(f"[工具執行] {tool_name} → {result}")
        return {"tools_result": result}
    return {"tools_result": "無需工具"}


# 整合版：含工具結果的回應
def full_tool_response(state: FullVoiceState) -> dict:
    text = state.get("normalized_text", "")
    context = state.get("context", "")
    tools_result = state.get("tools_result", "")
    prompt = f"""你是友善的客服語音助理。
對話歷史：{context}
使用者說：{text}
工具查詢結果：{tools_result}

根據工具結果回覆使用者。用繁體中文，語氣親切。"""
    result = llm.invoke(prompt)
    print(f"[工具回應] 生成完成")
    return {"response_text": result.content.strip()}


# 整合版：信心度檢查
def full_confidence_check(state: FullVoiceState) -> dict:
    text = state.get("normalized_text", "")
    response = state.get("response_text", "")
    prompt = f"""評估回應的信心度（0.0-1.0）。
使用者：{text}
回應：{response}
只回傳數字。"""
    result = llm.invoke(prompt)
    try:
        confidence = float(result.content.strip())
    except:
        confidence = 0.5
    needs_human = confidence < 0.7
    print(f"[信心度] {confidence:.2f}, 需人工={needs_human}")
    return {"confidence": confidence, "needs_human": needs_human}


# 整合版：信心度路由
def full_confidence_route(state: FullVoiceState) -> str:
    if state.get("needs_human", False):
        return "human_review"
    return "memory_save"


# 整合版：人工審核
def full_human_review(state: FullVoiceState) -> dict:
    response = state.get("response_text", "")
    print(f"[人工審核] 信心度不足，轉交人工")
    return {"response_text": f"[已轉人工審核] {response}"}


# 整合版：記憶載入
def full_memory_load(state: FullVoiceState) -> dict:
    memory = state.get("memory", []) or []
    print(f"[記憶載入] {len(memory)} 筆")
    if memory:
        context = "\n".join([f"{m['role']}: {m['content']}" for m in memory[-6:]])
        return {"context": context}
    return {"context": "（無歷史對話）"}


# 整合版：記憶儲存
def full_memory_save(state: FullVoiceState) -> dict:
    memory = state.get("memory", []) or []
    new_memory = memory + [
        {"role": "user", "content": state.get("normalized_text", "")},
        {"role": "assistant", "content": state.get("response_text", "")},
    ]
    print(f"[記憶儲存] 共 {len(new_memory)} 筆")
    return {"memory": new_memory}

print("所有整合版節點定義完成")

In [ ]:
# === 建立完整整合圖 ===
full_graph = StateGraph(FullVoiceState)

# 加入所有節點
full_graph.add_node("stt", stt_node)
full_graph.add_node("normalize", normalize_node)
full_graph.add_node("memory_load", full_memory_load)
full_graph.add_node("classify_intent", classify_intent_node)
full_graph.add_node("tool_selection", full_tool_selection)
full_graph.add_node("tool_execution", full_tool_execution)
full_graph.add_node("tool_response", full_tool_response)
full_graph.add_node("complaint_response", full_complaint_response)
full_graph.add_node("general_response", full_general_response)
full_graph.add_node("confidence_check", full_confidence_check)
full_graph.add_node("human_review", full_human_review)
full_graph.add_node("memory_save", full_memory_save)
full_graph.add_node("tts", tts_node)

# 入口：STT → 正規化 → 記憶載入 → 意圖分類
full_graph.set_entry_point("stt")
full_graph.add_edge("stt", "normalize")
full_graph.add_edge("normalize", "memory_load")
full_graph.add_edge("memory_load", "classify_intent")

# Orchestrator 路由
full_graph.add_conditional_edges(
    "classify_intent",
    full_orchestrator_route,
    {
        "tool_selection": "tool_selection",
        "complaint_response": "complaint_response",
        "general_response": "general_response",
    }
)

# 工具路線：選擇 → 執行 → 回應
full_graph.add_edge("tool_selection", "tool_execution")
full_graph.add_edge("tool_execution", "tool_response")

# 所有回應路線匯入信心度檢查
full_graph.add_edge("tool_response", "confidence_check")
full_graph.add_edge("complaint_response", "confidence_check")
full_graph.add_edge("general_response", "confidence_check")

# 信心度路由
full_graph.add_conditional_edges(
    "confidence_check",
    full_confidence_route,
    {
        "human_review": "human_review",
        "memory_save": "memory_save",
    }
)

# 人工審核後也存記憶
full_graph.add_edge("human_review", "memory_save")

# 記憶儲存 → TTS → 結束
full_graph.add_edge("memory_save", "tts")
full_graph.add_edge("tts", END)

# 編譯
full_app = full_graph.compile()
print("完整整合圖建立完成！")

In [ ]:
# 視覺化完整圖
try:
    display(Image(full_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"視覺化需要額外套件，跳過: {e}")

In [ ]:
# 挑戰 1 起手式：情感分析節點
# 請在此實作

def sentiment_analysis_node(state):
    """分析使用者情感"""
    text = state.get("normalized_text", "")
    # TODO: 用 LLM 分析情感
    # prompt = ...
    # result = llm.invoke(prompt)
    pass

In [ ]:
# 挑戰 2 起手式：升級處理邏輯
# 請在此實作

def check_escalation_node(state):
    """檢查是否需要升級處理"""
    memory = state.get("memory", []) or []
    # TODO: 計算連續客訴次數
    # 如果超過 3 次，設定 needs_escalation = True
    pass


def escalation_node(state):
    """升級處理：轉接主管"""
    # TODO: 呼叫 transfer_to_agent
    pass